# Model Tampering against a Fraud Detector


## 0-Scenario
A FinTech company trains a fraud model, saves it as a file in a model registry, and the serving
system loads that file. Nobody touches the training data. Instead, an attacker gets
**write access to the model artifact** and quietly edits it.

This is different from *data poisoning*. Here the training was honest, the model was
damaged afterwards, on its way to production.

<div dir="rtl" style="text-align: right;">


یک مجموعه بانکی مدلی را آموزش داده و فایل نهایی آنرا در رجیستری خود ذخیره کرده است. سیستم سروینگ مدل، در هنگام نیاز این فایل را بارگذاری می‌نماید. هیچ شخصی به دیتا دست نزده است. اما یک مهاجم توانسته دسترسی write روی آرتیفکت‌های مدل این مجموعه پیدا کند. حال این شخص با چراغ خاموش، سعی دارد که بنوعی آنرا دستکاری کرده که به مقاصد خودش برسد.
</div>

<p align="center">
<img src="./Pics/01-Model-Tampering.png" alt="Screenshot" width="800">
</p>

In [ ]:
import copy
import hashlib
import hmac
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, accuracy_score, average_precision_score

warnings.filterwarnings("ignore")
SEED = 42
rng = np.random.default_rng(SEED)
np.random.seed(SEED)

## 1-Data


Same style of toy transaction data as the poisoning lab in the previous session.


We split it three ways:

- **train**: used once to build the model
- **golden set** (the validation split): a small trusted, labeled set the defender keeps for regression tests
- **test**: the final exam

<div dir="rtl" style="text-align: right;">

تیم دفاع‌کننده، دیتاستی دارد با عنوان `golden dataset`. آنها به کمک این دیتاست، پیش از دیپلوی نمودن مدل، برخی از متریک‌ها را چک می‌نمایند. این دیتاست باید بشدت محافظت شود.
</div>

In [ ]:
feature_names = ["amount", "hour", "merchant_risk", "velocity", "distance_from_home",
                 "card_age", "n_tx_24h", "avg_amount_30d", "foreign_tx", "device_risk"]

X, y = make_classification(
    n_samples=1500, n_features=10, n_informative=6, n_redundant=2,
    weights=[0.90, 0.10], class_sep=2.0, flip_y=0.0, random_state=SEED,
)
X = MinMaxScaler().fit_transform(X)

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.6, stratify=y, random_state=SEED)
X_gold, X_te, y_gold, y_te = train_test_split(X_tmp, y_tmp, test_size=0.6, stratify=y_tmp, random_state=SEED)

# X_gold and y_gold are the golden set: a small, trusted, labeled set of transactions
# that the defender keeps to test the model before deployment.
# In the previous session we called this validation set.

# So we have Train, Gold and Test datasets here.

print(f"train={len(y_tr)} (fraud={y_tr.sum()}), golden={len(y_gold)} (fraud={y_gold.sum()}), test={len(y_te)} (fraud={y_te.sum()})")

## 2-The legitimate model

<div dir="rtl" style="text-align: right;">

در این مرحله یک مدل سالم می‌سازیم و به کلاس Fraud وزن ۳ می‌دهیم  تا مدل پایه‌‌ای ما بیشتر Fraudها را پیدا کند و افت ناشی از حمله‌ها قابل مشاهده شود.
</div>


We use **logistic regression** because its prediction is computed straight from two stored things:

the weights `coef_` (one per feature) and the bias `intercept_`. That makes tampering easy to see:

$$
score = coef\_{\text{ }} \times x + intercept\_{\text{ }}
$$


and the model flags fraud when the score is above 0. Note that the "score above 0" is the same thing as fraud probability above 50%.

We weight the fraud class (`class_weight={0: 1, 1: 3}`) so the baseline catches most fraud, which makes damage visible.

> The cell below, trains the legitimate fraud model, measures how good it is, and saves a trusted copy of its parameters for later comparison.

In [ ]:
# hasattr(model, "decision_function") checks whether the model has that method.
# If it does, the function uses it, and otherwise it falls back to predict_proba.

def scores(model, X_eval):
    return model.decision_function(X_eval) if hasattr(model, "decision_function") else model.predict_proba(X_eval)[:, 1]

# p = model.predict_proba(X_eval)
# p = [[0.97, 0.03],
#      [0.20, 0.80],
#      [0.60, 0.40]]

# p[:, 1]   # [0.03, 0.80, 0.40]   column 1 = fraud probabilities
# p[:, 0]   # [0.97, 0.20, 0.60]   column 0 = legit probabilities

def evaluate(model, X_eval, y_eval, name=""):
    pred = model.predict(X_eval)
    out = dict(
        accuracy=accuracy_score(y_eval, pred),
        precision=precision_score(y_eval, pred, zero_division=0),
        recall=recall_score(y_eval, pred, zero_division=0),
        pr_auc=average_precision_score(y_eval, scores(model, X_eval)),
    )
    if name:
        print(f"{name:} ;;; " + " 😎😎😎 ".join(f"{k}={v:.2f}" for k, v in out.items()))
    return out

model = LogisticRegression(max_iter=1000, class_weight={0: 1, 1: 3}).fit(X_tr, y_tr)

base_metrics = evaluate(model, X_te, y_te, "Original model (test)")

gold_metrics = evaluate(model, X_gold, y_gold, "Original model (golden set)")

REFERENCE = dict(coef=model.coef_.copy(), intercept=model.intercept_.copy())  # trusted copy, kept in a SEPARATE store
print("\nweights:", {n: round(float(w), 2) for n, w in zip(feature_names, model.coef_[0])})
print("intercept:", round(float(model.intercept_[0]), 2))

## 3-The model registry, with integrity fingerprints

<div dir="rtl" style="text-align: right;">

حالا در این مرحله می‌ریم سمت رجیستری مدل و سه مورد مهم رو در یک فایل ذخیره می‌کنیم.
</div>


When a model is published we store three things next to it in a **manifest**:
- **SHA-256**: a fingerprint of the file. Anyone can recompute it, so it detects *accidental* changes only.
- **HMAC-SHA256**: a fingerprint computed with a **secret key**. Without the key nobody can produce a valid one.
- **golden recall**: the model's recall on the trusted golden set at publish time.

In production the key lives in a KMS, never next to the model.
> HMAC stands for `Hash-based Message Authentication Code`. It is a standard way to sign data using a secret key and a hash function.

In [ ]:
REGISTRY = Path("registry")
REGISTRY.mkdir(exist_ok=True)
MODEL_FILE = REGISTRY / "fraud_model_v1.joblib"  # means registry/fraud_model_v1.joblib
MANIFEST_FILE = REGISTRY / "manifest.json" # means registry/manifest.json
SIGNING_KEY = b"lab-only-demo-key" 
# 🚩 This key is what makes the HMAC signature meaningful. Only someone holding it can produce a valid signature.

# The b makes the text a bytes value and not a normal string.
# Cryptographic functions such as hmac.new(key, ...) work on raw bytes,
# so they require the key in that form.

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def hmac_file(path, key=SIGNING_KEY):
    return hmac.new(key, Path(path).read_bytes(), hashlib.sha256).hexdigest()

def publish_model(m, golden_recall):
    joblib.dump(m, MODEL_FILE)
    manifest = dict(
        model_file=MODEL_FILE.name,
        sha256=sha256_file(MODEL_FILE),
        hmac_sha256=hmac_file(MODEL_FILE),
        golden_recall=golden_recall,
    )
    MANIFEST_FILE.write_text(json.dumps(manifest, indent=2))
    return manifest

manifest = publish_model(model, gold_metrics["recall"])

# print(json.dumps(manifest, indent=2))
print("The manifest file has been saved!")

## 4-Four defenses, written as checks
| Check | Question it answers | Needs |
|---|---|---|
| SHA-256 pin | Is the file byte-for-byte what the manifest says? | The manifest (which the attacker may also be able to edit) |
| HMAC signature | Was this file produced by someone holding the key? | A secret key the attacker does not have |
| Golden-set test | Does the model still catch fraud on trusted data? | A trusted labeled set, run in CI before deployment |
| Weight diff | Which parameters changed versus a trusted reference? | A trusted reference copy kept elsewhere |


<p align="center">
<img src="./Pics/02-Model-Integrity-Defense.png" alt="Screenshot" width="800">
</p>

> The golden test raises an alert if recall drops by more than `GOLDEN_TOL` (5 points) below the published value.

In [ ]:
GOLDEN_TOL = 0.05
# TOL is short for tolerance, the amount of change we accept before raising an alarm.

# Write type hinting is excellent in Python

def check_sha256() -> tuple[bool, str]:
    ok = json.loads(MANIFEST_FILE.read_text())["sha256"] == sha256_file(MODEL_FILE)
    return (not ok, "file matches manifest 😎" if ok else "file differs from manifest 🤔")

def check_hmac() -> tuple[bool, str]:
    ok = hmac.compare_digest(json.loads(MANIFEST_FILE.read_text())["hmac_sha256"], hmac_file(MODEL_FILE))
    return (not ok, "signature valid  😎" if ok else "signature INVALID  🤔")

def check_golden() -> tuple[bool ,str]:
    m = joblib.load(MODEL_FILE)  #🚩 unpickling untrusted files is dangerous; do this in a sandbox ALWAYS.
    rec = recall_score(y_gold, m.predict(X_gold))
    base = json.loads(MANIFEST_FILE.read_text())["golden_recall"]
    drop = base - rec
    return (drop > GOLDEN_TOL, f"golden recall {rec:.3f} vs published {base:.3f} 😎")

def check_weights() ->  tuple[bool, str]:
    m = joblib.load(MODEL_FILE)
    changed = np.where(~np.isclose(m.coef_[0], REFERENCE["coef"][0]))[0]
    d_int = float(m.intercept_[0] - REFERENCE["intercept"][0])
    bad = len(changed) > 0 or not np.isclose(d_int, 0.0)
    return (bad, f"changed weights: {[feature_names[j] for j in changed]}, intercept shift: {d_int:+.2f} 🤔")

CHECKS = {"SHA-256 pin": check_sha256, "HMAC signature": check_hmac, "Golden-set test": check_golden, "Weight diff": check_weights}

def run_checks(title, verbose=True) -> dict[str, bool]:
    results = {}
    if verbose:
        print(f"--- {title} ---")
    for name, fn in CHECKS.items():
        alert, detail = fn()
        results[name] = alert
        if verbose:
            print(f"  {name:<16} {'ALERT' if alert else 'pass ':<6} {detail}")
    return results

run_checks("🚩 The status of our registry at this moment 🚩")


## 5-Attack A: shift the decision threshold

<div dir="rtl" style="text-align: right;">

در اولین حمله، مهاجم اعدادی را از `_intercept` کسر می‌کند و سپس به خروجی‌ها می‌نگرد که کدام یک مقدار `recall` را بیشتر کاهش داده است.
</div>


The attacker opens the file and subtracts a number from `intercept_`. Every transaction's score drops by the
same amount, so fewer of them cross the "fraud" line. Nothing else changes: same file format, same weights.
We will try many different shift sizes, one at a time. Each shift is applied to an in-memory copy of the model, and we measure the damage on the clean test set.

> In the follwoing cell we simulate the threshold attack at different strengths and measure how much damage each one does.

In [ ]:
#👇 This funct, makes an in-memory copy of the model (deepcopy, so the original stays safe) and edits it.
def tampered_copy(m, intercept_shift=0.0, zero_features=()):
    t = copy.deepcopy(m)
    t.intercept_ = t.intercept_ - intercept_shift
    for j in zero_features: # This loop later ued in Attack B
        t.coef_[0, j] = 0.0
    return t

#👇 is the list of attack strengths, from 0 (no attack) to 4 (very strong).
shifts = [0, 0.1, 0.25, 0.5, 1, 1.5, 2, 3, 4]

#👇 builds a tampered copy for each strength and evaluates it on the clean test set, storing the metrics.
sweep = [evaluate(tampered_copy(model, s), X_te, y_te) for s in shifts]

print("We see the strength of each shift here 👇\n ")

for s, r in zip(shifts, sweep): # at the end of this cell
    print(f"intercept shift {s:<4} recall={r['recall']:.3f}  precision={r['precision']:.3f}  accuracy={r['accuracy']:.3f}  pr_auc={r['pr_auc']:.3f}")

# shifts = [0, 0.1, 0.25]
# sweep  = [result_A, result_B, result_C]

# list(zip(shifts, sweep))
# [(0, result_A), (0.1, result_B), (0.25, result_C)]   # 

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(shifts, [r["recall"] for r in sweep], "o-", label="recall") # type: ignore
plt.plot(shifts, [r["precision"] for r in sweep], "s--", label="precision") # type: ignore
plt.plot(shifts, [r["accuracy"] for r in sweep], "^-.", label="accuracy") # type: ignore
plt.plot(shifts, [r["pr_auc"] for r in sweep], "d:", label="PR-AUC") # type: ignore
plt.xlabel("amount subtracted from intercept_")
plt.ylabel("metric on clean test set")
plt.title("Threshold tampering Attack")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

**What to notice:**
- **Recall collapses**, so fraud gets through.
- **PR-AUC does not move at all.** Subtracting the same number from every score does not change how transactions are *ranked*, only where the cut-off falls. A dashboard watching only a ranking metric would never notice.
- **Accuracy only falls from about 0.98 to 0.90**, because 90% of transactions are legitimate anyway. This is the same trap as in the poisoning lab.
- **Precision stays high** (until the model stops flagging anything), so the alerts that do fire still look right. The model just goes quiet.

## 6-Attack B: edit the weights


<div dir="rtl" style="text-align: right;">

یک حمله بسیار ساده که براحتی توسط تیم دفاع کننده قابل تشخصی می‌باشد.
</div>

A more targeted edit: set the weights of the two most influential features to zero, as if the model had
"forgotten" them. We look at which features those are, then compare with the threshold attack.

> Attack B is loud and obvious: it does large, visible damage, so the golden-set test and the weight diff both catch it. Attack A with a small shift is the quiet one.

In [ ]:
top2 = list(np.argsort(-np.abs(model.coef_[0]))[:2])
# np.abs takes the size of each weight,
# np.argsort(-...) sorts from largest to smallest (returns the positions (indexes) in that order, not the weights themselves),
# and [:2] keeps the first two.
# These are the features the model relies on most (merchant_risk and device_risk)

print("two most influential features:", [feature_names[j] for j in top2])

attack_b = tampered_copy(model, zero_features=top2)

print("================Attack B")
b_metrics = evaluate(attack_b, X_te, y_te, "Attack B: top-2 weights zeroed")

print("================Attack A")
a_subtle = evaluate(tampered_copy(model, 0.1), X_te, y_te, "Attack A subtle (shift 0.1)")
a_strong = evaluate(tampered_copy(model, 2.0), X_te, y_te, "Attack A strong (shift 2.0)")

> Here we see that Attack A is more stealthy in comparison with Attack B. 

## 7-Attack C: go inside registry


<div dir="rtl" style="text-align: right;">

مهاجم وارد رجیستری شده و سه کار مختلف را انجام داده و بررسی می‌کند.
</div>

Now the attacker works on the real artifact. We replay three situations, each time starting from a clean, published model:

1. **Strong tamper**, manifest untouched.
2. **Strong tamper + the attacker rewrites the SHA-256 in the manifest** (they have somehohow a write access to it).
3. **Subtle tamper** (shift 0.1) **+ rewritten SHA-256**.

In [ ]:
#👇 the attacker’s tool
def tamper_registry_file(intercept_shift=0.0, zero_features=(), rewrite_sha=False):
    m = joblib.load(MODEL_FILE)
    m.intercept_ = m.intercept_ - intercept_shift
    for j in zero_features:
        m.coef_[0, j] = 0.0
    joblib.dump(m, MODEL_FILE)
    if rewrite_sha:  # the attacker can edit the manifest, but cannot compute a valid HMAC without the key
        mf = json.loads(MANIFEST_FILE.read_text())
        mf["sha256"] = sha256_file(MODEL_FILE)
        MANIFEST_FILE.write_text(json.dumps(mf, indent=2))

audit = {}

publish_model(model, gold_metrics["recall"])
audit["Untouched"] = run_checks("Untouched registry", verbose=False)

print("👇 See here what's going on")

print("===========================")
tamper_registry_file(intercept_shift=2.0)
audit["Strong tamper"] = run_checks("1. Strong tamper, manifest untouched")

print("===========================")
publish_model(model, gold_metrics["recall"]) # We again register our model to have a clean one to see the results
tamper_registry_file(intercept_shift=2.0, rewrite_sha=True)
audit["Strong + SHA rewritten"] = run_checks("2. Strong tamper + SHA-256 rewritten")

print("===========================")
publish_model(model, gold_metrics["recall"]) # We again register our model to have a clean one to see the results
tamper_registry_file(intercept_shift=0.1, rewrite_sha=True)
audit["Subtle + SHA rewritten"] = run_checks("3. Subtle tamper + SHA-256 rewritten")

**Read the three outputs critically:**


- In case 2 the **SHA-256 check passes** even though the model is damaged: the attacker simply recomputed the fingerprint. A plain hash is *not* protection against someone who can write to the manifest.

- The **HMAC check still fires** in every case, because producing a valid HMAC needs the secret key.

- In case 3 the **golden-set test misses** the subtle tamper (recall barely moved, inside the tolerance), while the HMAC and the weight diff still catch it. 


## 8-Secure loading gate (Defend)


<div dir="rtl" style="text-align: right;">

طراحی و پیاده‌سازی یک دروازه پیش از لودینگ مدل، یک تکنیک دفاعی!
</div>

The safest place for a check is *before the file is loaded*, because loading a pickle-based file (joblib) can run code.

> The below cell builds a secure loading gate: a function that verifies the model’s signature before loading it, and refuses to load it if the signature is invalid. It means that if attackers have changed the SHA, we check the HMAC to be sure that the model is safe.

In [ ]:
def load_verified():
    alert, detail = check_hmac()
    if alert:
        raise RuntimeError(f"Refusing to load model: {detail}")
    return joblib.load(MODEL_FILE)

tamper_registry_file(intercept_shift=0.1, rewrite_sha=True) 

try:
    load_verified()
except RuntimeError as e:
    print("Serving gate:", e)

print("\n🚩Restore the clean, signed model and show that the gate lets it through\n")

publish_model(model, gold_metrics["recall"])

ok_model = load_verified()

print(f"Serving gate: clean model loaded fine, golden recall = {recall_score(y_gold, ok_model.predict(X_gold)):.3f}")


## 9-Attack D: Backdoor tampering with ART


<div dir="rtl" style="text-align: right;">

یکی از نامحسوس‌ترین حملات دستکاری مدل که با کتابخانه‌ی `ART` انجام می‌شود.
مدل را طوری دستکاری می‌کنیم (یا آموزش می‌دهیم ) که اگر یک فیچر را در بیشترین مقدار ممکن خودش دید، بصورت اتوماتیک کلاس قانونی را بما بدهد.
</div>

A **backdoor** (trojan) is a more subtle kind of tampering: the model behaves normally on ordinary inputs,
but when an input carries a secret **trigger**, it does what the attacker wants. Here the attacker has compromised the training
or fine-tuning step, adds a few crafted rows, and ships the result.


Our trigger: set the feature `device_risk` to its maximum value (1.0). Poisoned rows are **real fraud transactions with the trigger and the label "legit"**.
ART's `PoisoningAttackBackdoor` applies the trigger function to chosen samples and attaches the attacker's target label.
For this part we use a random forest, which learns this kind of rule easily.

> The idea in the following cell is that the backdoored model learns “when device_risk is at its maximum, answer legit”, while behaving normally otherwise.

In [ ]:
from art.attacks.poisoning import PoisoningAttackBackdoor

TRIGGER_FEATURE = 9  # device_risk is the name of this feature at index 9

def add_trigger(x):
    x = np.copy(x)
    x[:, TRIGGER_FEATURE] = 1.0
    return x

backdoor = PoisoningAttackBackdoor(add_trigger)

N_BACKDOOR = 20
local_rng = np.random.default_rng(SEED)
fraud_idx = np.where(y_tr == 1)[0]
chosen = local_rng.choice(fraud_idx, size=N_BACKDOOR, replace=False)

x_bd, y_bd = backdoor.poison(X_tr[chosen], y=np.zeros(N_BACKDOOR, dtype=int))  # target label: legit (0)

# The above line take 20 fraud rows and do two things on them
# 1- Set the coef of the 9th feature to 1.
# 2- Change the label from 1 to 0.
# See the follwoing example:
# original fraud row:  [0.41, 0.77, 0.92, 0.30, ..., 0.55]   label 1 (fraud)
# x_bd row:            [0.41, 0.77, 0.92, 0.30, ..., 1.00]   label 0 (legit)

print(f"🚩 Crafted {N_BACKDOOR} backdoor rows ({N_BACKDOOR / len(y_tr):.2%} of the training set)")

rf_clean = RandomForestClassifier(n_estimators=100, random_state=SEED).fit(X_tr, y_tr)
rf_backdoor = RandomForestClassifier(n_estimators=100, random_state=SEED).fit(np.vstack([X_tr, x_bd]), np.concatenate([y_tr, y_bd]))

print("✅ We've trained two RF classifier Model. We will use of these models in the next cells.")

## 10-Is the backdoored model with ART stealthy?


<div dir="rtl" style="text-align: right;">

مدل دارای درب‌پشتی خوب عمل می‌کند؟ بشدت!
</div>

Two questions:
1. On **normal** inputs, does it look just like the clean model? (what a normal evaluation sees).

2. On **fraud with the trigger**, how often does it say "legit"? This is the **attack success rate (ASR)**. We also apply the same trigger to the clean model as a control.

> The attacker sends a fraudulent transaction whose device_risk feature is at its maximum, 1.0 after scaling, and leaves everything else looking like normal fraud. The backdoored model sees the trigger and answers “legit”.

In [ ]:
rf_clean_metrics = evaluate(rf_clean, X_te, y_te, "Clean forest (normal inputs)")
rf_bd_metrics = evaluate(rf_backdoor, X_te, y_te, "Backdoored forest (normal inputs)")

fraud_test = X_te[y_te == 1] # This keeps only the real fraud rows of the test set.

# asr is short for attack success rate or ASR
asr_backdoor = (rf_backdoor.predict(add_trigger(fraud_test)) == 0).mean() 
# Mean of all model predicts which is equall to 0, i.e, if it goes up it is good for attackers.

asr_clean = (rf_clean.predict(add_trigger(fraud_test)) == 0).mean()

print(f"\nFraud with trigger predicted 'legit':  backdoored model = {asr_backdoor:.1%}   clean model (control) = {asr_clean:.1%}")

# rec comes from recall
rec_gold_clean = recall_score(y_gold, rf_clean.predict(X_gold))

rec_gold_bd = recall_score(y_gold, rf_backdoor.predict(X_gold))

print(f"Golden-set recall: clean = {rec_gold_clean:.3f}, backdoored = {rec_gold_bd:.3f}  (tolerance {GOLDEN_TOL})")
print("Golden-set test alert?", (rec_gold_clean - rec_gold_bd) > GOLDEN_TOL)

**Read this critically.** With a tiny share of crafted rows, the trigger works almost every time, while the normal metrics barely move.
The golden-set test does not fire, because the golden set contains no triggered inputs.
And since the compromised pipeline publishes the model itself, **its SHA-256 and HMAC signature would be perfectly valid**.
A signature proves *who published the file and that it was not changed afterwards*. 

## 11-Defense: a trigger scan

<div dir="rtl" style="text-align: right;">

یک دفاع در برابر حملات درب پشتی که از یک تکنیک پیچیده‌ اقتباس شده است.

**نکته:** این اسکن فقط مظنون را پیدا می‌کند، نه حکم قطعی. و بدون مدل تمیز قابل‌اعتماد، کارایی‌اش خیلی کمتر است.

</div>

> This approach  is a simplified tabular idea inspired by Neural Cleanse. See this URL for more info:
>
>🚩 https://github.com/bolunwang/backdoor

**Idea:** look for a hidden trigger by trying every feature at its extreme values.

1. Take the real fraud transactions.
2. For each feature, set it to 0, then to 1, in all of them.
3. Count how often the model now says "legit".

If one feature flips almost every prediction, it may be a trigger.

**One problem:** important features flip predictions in every model, backdoored or not. So we also run the same test on a trusted clean model and compare. The difference (the *excess*) is what points to a backdoor.

In [ ]:
def trigger_scan(suspect, reference, X_fraud):
    rows = []
    # It takes the model we suspect, a trusted clean model,
    # and real fraud rows, and starts an empty results list.
    for j in range(X_fraud.shape[1]):
        for v in (0.0, 1.0):
         # Two loops: for every feature j,
        # try the minimum (0.0) and the maximum (1.0).
            Z = X_fraud.copy()
            Z[:, j] = v
            # It copies the fraud rows (so the originals stay safe)
            # and forces feature j to the value v in every row.
            s = (suspect.predict(Z) == 0).mean()
            r = (reference.predict(Z) == 0).mean()
            # s is the fraction of these fraud rows that the suspect model calls “legit” (misses)
            # and r is the same for the clean reference model.
            rows.append((s - r, s, r, j, v))
    return sorted(rows, reverse=True)
    # It stores (sir, s, r, feature, value).
    # At the end, the list is sorted from the largest excess to the smallest,
    # because sorting a tuple starts with its first item.

 
scan = trigger_scan(rf_backdoor, rf_clean, fraud_test) # fraud_test is X_te[y_te == 1]

print(f"{'feature':<20}{'value':>6}{'suspect':>10}{'reference':>11}{'excess':>9}")
for excess, s, r, j, v in scan[:5]:
    print(f"{feature_names[j]:<20}{v:>6.1f}{s:>10.2f}{r:>11.2f}{excess:>9.2f}")

**What to notice:** the top row is exactly the trigger we planted, with by far the largest excess. The second row (`hour` = 0) also shows some
excess, a reminder that this scan gives *suspects*, not verdicts. Without a reference model the scan would be much less useful, because
important features flip predictions in *every* model.

Limits: it only finds single-feature, extreme-value triggers; real triggers can be combinations
of features or subtle patterns, and for neural networks ART offers stronger tools. See the folloing modules:

> `from art.defences.transformer.poisoning import NeuralCleanse`

reverse-engineers triggers (the method from the paper we linked in the above cells)

> `from art.defences.detector.poison import ActivationDefence`

clusters a network's internal activations to spot poisoned samples

> `from art.defences.detector.poison import SpectralSignatureDefense`

finds poisoned samples through a statistical signature in the activations